# 국가별 전력믹스 분석: GB vs HPB 비교

이 노트북은 세 국가(South Korea, China, Norway)를 선택하여 재생에너지 비율 증가에 따른 Gas Boiler와 Heat Pump Boiler의 성능을 비교합니다.

비교 지표:
- CO2 배출 계수
- Primary energy use
- Energy use
- Exergy efficiency


In [1]:
%load_ext autoreload
%autoreload 2

# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os
import dartwork_mpl as dm

# Apply dartwork_mpl style
dm.style.use('scientific')  # 한국어 논문용 스타일

# Add parent directory to path if needed
sys.path.append(os.path.dirname(os.path.abspath('')))

from grid_mix_analysis import (
    COUNTRY_GRID_MIX,
    DEFAULT_TECH_PARAMS,
    sweep_country_renewable_fraction,
    save_country_grid_analysis,
    load_country_grid_analysis,
    calculate_grid_co2_factor,
    calculate_grid_primary_energy_factor,
    calculate_grid_exergy_efficiency
)
from config import PAPER_DEFAULT_SCENARIO

print("Libraries imported successfully!")


Libraries imported successfully!


## GB와 HPB의 에너지 사용량 설정

실제 시스템 객체에서 계산된 값을 사용하거나, 논문의 기본값을 사용합니다.


In [2]:
# GB와 HPB의 에너지 사용량 설정 (W 단위)
# 실제 시스템 객체에서 계산된 값을 사용하거나 기본값 사용
# 예시 값 (실제 계산값으로 대체 가능)

# Gas Boiler: E_NG (천연가스 에너지 사용량)
gb_energy_use = 1000.0  # W (예시값, 실제로는 GB.E_NG 사용)

# Heat Pump Boiler: E_cmp + E_fan (압축기 + 팬 전력 사용량)
hpb_energy_use = 500.0  # W (예시값, 실제로는 HPB.E_cmp + HPB.E_fan 사용)

print(f"GB Energy Use: {gb_energy_use} W")
print(f"HPB Energy Use: {hpb_energy_use} W")


GB Energy Use: 1000.0 W
HPB Energy Use: 500.0 W


## 세 국가 선택 및 데이터 생성

다양한 재생에너지 비율을 가진 세 국가를 선택합니다:
- **South Korea**: 낮은 재생에너지 비율 (11%)
- **China**: 중간 재생에너지 비율 (56%)
- **Norway**: 높은 재생에너지 비율 (99%)


In [3]:
# 선택된 세 국가
selected_countries = ['South Korea', 'China', 'Norway']

# 각 국가의 현재 재생에너지 비율 확인
for country in selected_countries:
    if country in COUNTRY_GRID_MIX:
        data = COUNTRY_GRID_MIX[country]
        print(f"{country}:")
        print(f"  - Renewable fraction: {data['renewable']:.2%}")
        print(f"  - Coal: {data['coal']:.2%}, Gas: {data['gas']:.2%}, Nuclear: {data['nuclear']:.2%}")
        print(f"  - Source: {data['source']}")
        print()


South Korea:
  - Renewable fraction: 11.00%
  - Coal: 27.00%, Gas: 27.00%, Nuclear: 32.00%
  - Source: KEPCO/Ember 2024, South Korea power mix

China:
  - Renewable fraction: 37.00%
  - Coal: 55.00%, Gas: 3.00%, Nuclear: 5.00%
  - Source: IEA Global Energy Review 2025, China power mix

Norway:
  - Renewable fraction: 99.00%
  - Coal: 0.00%, Gas: 1.00%, Nuclear: 0.00%
  - Source: Ember Energy 2024, Statnett/IEA 2024, Norway electricity mix



## 국가별 그리드 분석 데이터 생성 및 CSV 저장

각 국가에 대해 renewable_fraction을 0부터 1까지 스윕하여 데이터를 생성하고 CSV로 저장합니다.


In [16]:
# Renewable fraction 범위 설정 (0부터 1까지 0.01 간격)
renewable_fractions = np.arange(0, 1.01, 0.01).tolist()

# 각 국가별로 데이터 생성 및 CSV 저장
output_dir = 'result'
os.makedirs(output_dir, exist_ok=True)

country_dataframes = {}

for country in selected_countries:
    print(f"Generating data for {country}...")
    
    # 데이터 생성
    df = sweep_country_renewable_fraction(
        country_name=country,
        renewable_fractions=renewable_fractions,
        gb_energy_use=gb_energy_use,
        hpb_energy_use=hpb_energy_use
    )
    
    # CSV 저장
    csv_path = save_country_grid_analysis(
        country_name=country,
        output_dir=output_dir,
        renewable_fractions=renewable_fractions,
        gb_energy_use=gb_energy_use,
        hpb_energy_use=hpb_energy_use
    )
    
    country_dataframes[country] = df
    print(f"  Saved to: {csv_path}\n")

print("All country data generated successfully!")


Generating data for South Korea...
Saved grid analysis for South Korea to result/South_Korea_grid_analysis.csv
  Saved to: result/South_Korea_grid_analysis.csv

Generating data for China...
Saved grid analysis for China to result/China_grid_analysis.csv
  Saved to: result/China_grid_analysis.csv

Generating data for Norway...
Saved grid analysis for Norway to result/Norway_grid_analysis.csv
  Saved to: result/Norway_grid_analysis.csv

All country data generated successfully!


## 비교 플롯 생성

각 지표별로 GB와 HPB를 비교하는 플롯을 생성합니다.


In [9]:
# GB의 CO2 배출 계수 (천연가스 기준)
gb_co2_factor = DEFAULT_TECH_PARAMS['gas'].co2_emission_factor  # kg CO2/kWh

# GB의 Primary Energy Factor (천연가스 기준)
gb_pef = DEFAULT_TECH_PARAMS['gas'].primary_energy_factor

# GB의 Exergy Efficiency (천연가스 기준)
gb_exergy_eff = DEFAULT_TECH_PARAMS['gas'].exergy_efficiency

print(f"GB Parameters:")
print(f"  - CO2 Factor: {gb_co2_factor:.4f} kg CO2/kWh")
print(f"  - Primary Energy Factor: {gb_pef:.2f}")
print(f"  - Exergy Efficiency: {gb_exergy_eff:.2%}")

# HPB는 전력망 믹스에 따라 변동하므로 각 국가별로 계산됨
print(f"\nHPB Parameters (vary with grid mix):")
print(f"  - CO2 Factor: varies with renewable fraction")
print(f"  - Primary Energy Factor: varies with renewable fraction")
print(f"  - Exergy Efficiency: varies with renewable fraction")


GB Parameters:
  - CO2 Factor: 0.4000 kg CO2/kWh
  - Primary Energy Factor: 1.80
  - Exergy Efficiency: 50.00%

HPB Parameters (vary with grid mix):
  - CO2 Factor: varies with renewable fraction
  - Primary Energy Factor: varies with renewable fraction
  - Exergy Efficiency: varies with renewable fraction


In [10]:
# 각 국가별로 계산된 값 준비
comparison_data = {}

for country in selected_countries:
    df = country_dataframes[country]
    
    # GB 값 (고정값)
    gb_co2_emission = gb_energy_use / 1000.0 * gb_co2_factor  # W를 kW로 변환 후 CO2 계산
    gb_primary_energy = gb_energy_use / 1000.0 * gb_pef  # W를 kW로 변환 후 PEF 적용
    
    # HPB 값 (재생에너지 비율에 따라 변동)
    hpb_co2_emission = hpb_energy_use / 1000.0 * df['co2_factor'].values  # W를 kW로 변환
    hpb_primary_energy = hpb_energy_use / 1000.0 * df['pef'].values  # W를 kW로 변환
    
    comparison_data[country] = {
        'renewable_fraction': df['renewable_fraction'].values,
        'gb_co2': gb_co2_emission,
        'hpb_co2': hpb_co2_emission,
        'gb_pe': gb_primary_energy,
        'hpb_pe': hpb_primary_energy,
        'gb_energy_use': gb_energy_use / 1000.0,  # kW
        'hpb_energy_use': hpb_energy_use / 1000.0,  # kW
        'gb_exergy_eff': gb_exergy_eff,
        'hpb_exergy_eff': df['exergy_efficiency'].values,
        'co2_factor': df['co2_factor'].values,
        'pef': df['pef'].values
    }

print("Comparison data prepared!")


Comparison data prepared!


In [11]:
# Figure 생성 (double column: 17cm 너비)
fig = plt.figure(
    figsize=(dm.cm2in(17), dm.cm2in(12)),
    dpi=200
)

# GridSpec 레이아웃 설정
gs = fig.add_gridspec(
    nrows=2, ncols=2,
    left=0.12, right=0.95,
    top=0.95, bottom=0.12,
    hspace=0.35, wspace=0.3
)

# 색상 설정 (dartwork_mpl 색상 사용)
colors = {
    'South Korea': 'oc.blue5',
    'China': 'oc.orange5', 
    'Norway': 'oc.green5'
}
gb_color = 'oc.red5'
gb_linestyle = '--'
hpb_linestyle = '-'
linewidth = 0.7

# 1. CO2 배출 계수 비교
ax1 = fig.add_subplot(gs[0, 0])
lines_hpb_co2 = []
for country in selected_countries:
    data = comparison_data[country]
    line, = ax1.plot(data['renewable_fraction'] * 100, data['co2_factor'],
                     linestyle=hpb_linestyle, linewidth=linewidth,
                     color=colors[country])
    lines_hpb_co2.append(line)

# GB는 고정값이므로 수평선
gb_line_co2 = ax1.axhline(y=gb_co2_factor, color=gb_color, 
                          linestyle=gb_linestyle, linewidth=linewidth)

ax1.set_xlabel('Renewable Energy Fraction [%]', fontsize=dm.fs(0))
ax1.set_ylabel('CO$_2$ Emission Factor [kg CO$_2$/kWh]', fontsize=dm.fs(0))
ax1.set_xlim(0, 100)
ax1.set_xticks([0, 25, 50, 75, 100])

# 레전드
legend_labels_co2 = [f'HPB ({country})' for country in selected_countries] + ['GB (Gas)']
ax1.legend(lines_hpb_co2 + [gb_line_co2], legend_labels_co2,
           fontsize=dm.fs(-1), loc='upper right', frameon=True)

# 2. Primary Energy Use 비교
ax2 = fig.add_subplot(gs[0, 1])
lines_hpb_pe = []
for country in selected_countries:
    data = comparison_data[country]
    line, = ax2.plot(data['renewable_fraction'] * 100, data['hpb_pe'],
                     linestyle=hpb_linestyle, linewidth=linewidth,
                     color=colors[country])
    lines_hpb_pe.append(line)

# GB는 고정값
gb_line_pe = ax2.axhline(y=comparison_data[selected_countries[0]]['gb_pe'],
                         color=gb_color, linestyle=gb_linestyle,
                         linewidth=linewidth)

ax2.set_xlabel('Renewable Energy Fraction [%]', fontsize=dm.fs(0))
ax2.set_ylabel('Primary Energy Use [kW]', fontsize=dm.fs(0))
ax2.set_xlim(0, 100)
ax2.set_xticks([0, 25, 50, 75, 100])

legend_labels_pe = [f'HPB ({country})' for country in selected_countries] + ['GB (Gas)']
ax2.legend(lines_hpb_pe + [gb_line_pe], legend_labels_pe,
           fontsize=dm.fs(-1), loc='upper right', frameon=True)

# 3. Energy Use 비교
ax3 = fig.add_subplot(gs[1, 0])
lines_hpb_eu = []
for country in selected_countries:
    data = comparison_data[country]
    # HPB energy use는 재생에너지 비율과 무관하게 일정
    line = ax3.axhline(y=data['hpb_energy_use'],
                      linestyle=hpb_linestyle, linewidth=linewidth,
                      color=colors[country])
    lines_hpb_eu.append(line)

# GB energy use도 일정
gb_line_eu = ax3.axhline(y=comparison_data[selected_countries[0]]['gb_energy_use'],
                         color=gb_color, linestyle=gb_linestyle,
                         linewidth=linewidth)

ax3.set_xlabel('Renewable Energy Fraction [%]', fontsize=dm.fs(0))
ax3.set_ylabel('Energy Use [kW]', fontsize=dm.fs(0))
ax3.set_xlim(0, 100)
ax3.set_xticks([0, 25, 50, 75, 100])

legend_labels_eu = [f'HPB ({country})' for country in selected_countries] + ['GB (Gas)']
ax3.legend(lines_hpb_eu + [gb_line_eu], legend_labels_eu,
           fontsize=dm.fs(-1), loc='upper right', frameon=True)

# 4. Exergy Efficiency 비교
ax4 = fig.add_subplot(gs[1, 1])
lines_hpb_ex = []
for country in selected_countries:
    data = comparison_data[country]
    line, = ax4.plot(data['renewable_fraction'] * 100, data['hpb_exergy_eff'] * 100,
                     linestyle=hpb_linestyle, linewidth=linewidth,
                     color=colors[country])
    lines_hpb_ex.append(line)

# GB는 고정값
gb_line_ex = ax4.axhline(y=gb_exergy_eff * 100, color=gb_color,
                         linestyle=gb_linestyle, linewidth=linewidth)

ax4.set_xlabel('Renewable Energy Fraction [%]', fontsize=dm.fs(0))
ax4.set_ylabel('Exergy Efficiency [%]', fontsize=dm.fs(0))
ax4.set_xlim(0, 100)
ax4.set_xticks([0, 25, 50, 75, 100])

legend_labels_ex = [f'HPB ({country})' for country in selected_countries] + ['GB (Gas)']
ax4.legend(lines_hpb_ex + [gb_line_ex], legend_labels_ex,
           fontsize=dm.fs(-1), loc='upper right', frameon=True)

# 서브플롯 라벨 추가 (a, b, c, d)
for idx, (ax, label) in enumerate(zip([ax1, ax2, ax3, ax4], 'abcd')):
    offset = dm.make_offset(4, -4, fig)
    ax.text(0, 1, f'({label})', transform=ax.transAxes + offset,
           weight='bold', verticalalignment='top', fontsize=dm.fs(1))

# 레이아웃 최적화
dm.simple_layout(fig)

# 저장 및 표시
os.makedirs('figure', exist_ok=True)
# dm.save_formats(
#     fig,
#     'figure/country_comparison_gb_vs_hpb',
#     formats=('svg', 'png', 'pdf'),
#     bbox_inches='tight',
#     dpi=300
# )
dm.save_and_show(fig, size=600)


## 추가 분석: 각 국가별 상세 비교

각 국가별로 CO2 배출과 Primary Energy를 더 자세히 비교합니다.
- 왜. renewable 비율이 늘어나는데 primary energy 비중이 늘어나는거지

In [12]:
# 각 국가별로 상세 비교 플롯 생성
fig = plt.figure(
    figsize=(dm.cm2in(17), dm.cm2in(18)),
    dpi=200
)

# GridSpec 레이아웃 설정
gs = fig.add_gridspec(
    nrows=3, ncols=2,
    left=0.12, right=0.95,
    top=0.97, bottom=0.10,
    hspace=0.35, wspace=0.3
)

hpb_color = 'oc.blue5'
gb_color = 'oc.red5'
linewidth = 0.7

for idx, country in enumerate(selected_countries):
    data = comparison_data[country]
    
    # CO2 배출 비교
    ax_co2 = fig.add_subplot(gs[idx, 0])
    hpb_line_co2, = ax_co2.plot(data['renewable_fraction'] * 100, data['hpb_co2'],
                                color=hpb_color, linewidth=linewidth, label='HPB')
    gb_line_co2 = ax_co2.axhline(y=data['gb_co2'], color=gb_color,
                                 linestyle='--', linewidth=linewidth, label='GB')
    
    ax_co2.set_xlabel('Renewable Energy Fraction [%]', fontsize=dm.fs(0))
    ax_co2.set_ylabel('CO$_2$ Emission [kg CO$_2$/h]', fontsize=dm.fs(0))
    ax_co2.set_xlim(0, 100)
    ax_co2.set_xticks([0, 25, 50, 75, 100])
    ax_co2.legend([hpb_line_co2, gb_line_co2], ['HPB', 'GB'],
                  fontsize=dm.fs(-1), loc='upper right', frameon=True)
    
    # 국가명 추가
    offset = dm.make_offset(4, -4, fig)
    ax_co2.text(0, 1, country, transform=ax_co2.transAxes + offset,
               weight='bold', verticalalignment='top', fontsize=dm.fs(1))
    
    # Primary Energy 비교
    ax_pe = fig.add_subplot(gs[idx, 1])
    hpb_line_pe, = ax_pe.plot(data['renewable_fraction'] * 100, data['hpb_pe'],
                             color=hpb_color, linewidth=linewidth, label='HPB')
    gb_line_pe = ax_pe.axhline(y=data['gb_pe'], color=gb_color,
                              linestyle='--', linewidth=linewidth, label='GB')
    
    ax_pe.set_xlabel('Renewable Energy Fraction [%]', fontsize=dm.fs(0))
    ax_pe.set_ylabel('Primary Energy Use [kW]', fontsize=dm.fs(0))
    ax_pe.set_xlim(0, 100)
    ax_pe.set_xticks([0, 25, 50, 75, 100])
    ax_pe.legend([hpb_line_pe, gb_line_pe], ['HPB', 'GB'],
                 fontsize=dm.fs(-1), loc='upper right', frameon=True)

# 레이아웃 최적화
dm.simple_layout(fig)

# 저장 및 표시
dm.save_formats(
    fig,
    'figure/detailed_country_comparison',
    formats=('svg', 'png', 'pdf'),
    bbox_inches='tight',
    dpi=300
)
dm.save_and_show(fig, size=600)


## Critical Point 분석

각 국가별로 HPB가 GB보다 유리해지는 재생에너지 비율(critical point)을 찾습니다.


In [ ]:
from grid_mix_analysis import find_critical_point

# Critical point 계산
critical_points = {}

for country in selected_countries:
    data = comparison_data[country]
    
    # GB 값 (고정)
    gb_pe = comparison_data[country]['gb_pe']
    gb_co2 = comparison_data[country]['gb_co2']
    
    # HPB 값 (재생에너지 비율에 따라 변동)
    hpb_pe_base = comparison_data[country]['hpb_energy_use']  # 기본 전력 사용량
    hpb_co2_base = comparison_data[country]['hpb_energy_use'] * comparison_data[country]['co2_factor'][0]  # 초기 CO2
    
    # Critical point 찾기
    critical = find_critical_point(
        hpb_primary_energy  = hpb_pe_base,
        gb_primary_energy   = gb_pe,
        hpb_co2             = hpb_co2_base,
        gb_co2              = gb_co2,
        renewable_fractions = data['renewable_fraction'],
        pef_values          = data['pef'],
        co2_values          = data['co2_factor']
    )
    
    critical_points[country] = critical
    
    print(f"{country}:")
    if 'pe_critical_renewable_fraction' in critical:
        print(f"  - PE Critical Point: {critical['pe_critical_renewable_fraction']:.2%} renewable")
        print(f"    (PEF = {critical['pe_critical_pef']:.3f})")
    else:
        print(f"  - PE Critical Point: Not found (HPB always better/worse)")
    
    if 'co2_critical_renewable_fraction' in critical:
        print(f"  - CO2 Critical Point: {critical['co2_critical_renewable_fraction']:.2%} renewable")
        print(f"    (CO2 Factor = {critical['co2_critical_co2_factor']:.4f} kg CO2/kWh)")
    else:
        print(f"  - CO2 Critical Point: Not found (HPB always better/worse)")
    print()


South Korea:
  - PE Critical Point: Not found (HPB always better/worse)
  - CO2 Critical Point: Not found (HPB always better/worse)

China:
  - PE Critical Point: Not found (HPB always better/worse)
  - CO2 Critical Point: Not found (HPB always better/worse)

Norway:
  - PE Critical Point: Not found (HPB always better/worse)
  - CO2 Critical Point: Not found (HPB always better/worse)



## Well-to-Tap 분석

Power generation부터 grid를 거쳐 최종 보일러까지의 전체 에너지 흐름을 분석합니다.
- Primary energy input
- Exergy consumption (generation, grid, total)
- CO2 emissions


In [17]:
# Import boiler systems and well-to-tap analyzer
import sys
sys.path.append(os.path.join(os.path.dirname(os.path.abspath('')), '..', '..', 'src'))

from enex_analysis.enex_engine import ElectricBoiler, GasBoiler, HeatPumpBoiler
from well_to_tap import WellToTapAnalyzer
from grid_mix_analysis import PowerGenerationMix

print("Boiler systems and WellToTapAnalyzer imported successfully!")


Boiler systems and WellToTapAnalyzer imported successfully!


### 보일러 시스템 초기화 및 계산

config.py의 설정을 사용하여 보일러 시스템 객체를 초기화하고 실제 에너지 사용량을 계산합니다.


In [18]:
# Initialize boiler systems using config
scenario = PAPER_DEFAULT_SCENARIO

# Electric Boiler
EB = ElectricBoiler()
EB.T0 = scenario.eb_config.temperature.T0
EB.T_w_tank = scenario.eb_config.temperature.T_w_tank
EB.T_w_sup = scenario.eb_config.temperature.T_w_sup
EB.T_w_serv = scenario.eb_config.temperature.T_w_serv
EB.dV_w_serv = scenario.eb_config.load.dV_w_serv
EB.r0 = scenario.eb_config.tank.r0
EB.H = scenario.eb_config.tank.H
EB.x_shell = scenario.eb_config.tank.x_shell
EB.x_ins = scenario.eb_config.tank.x_ins
EB.k_shell = scenario.eb_config.tank.k_shell
EB.k_ins = scenario.eb_config.tank.k_ins
EB.h_o = scenario.eb_config.tank.h_o
EB.system_update()

# Gas Boiler
GB = GasBoiler()
GB.eta_comb = scenario.gb_config.eta_comb
GB.T0 = scenario.gb_config.temperature.T0
GB.T_w_tank = scenario.gb_config.temperature.T_w_tank
GB.T_w_sup = scenario.gb_config.temperature.T_w_sup
GB.T_w_serv = scenario.gb_config.temperature.T_w_serv
GB.T_exh = scenario.gb_config.temperature.T_exh
GB.dV_w_serv = scenario.gb_config.load.dV_w_serv
GB.r0 = scenario.gb_config.tank.r0
GB.H = scenario.gb_config.tank.H
GB.x_shell = scenario.gb_config.tank.x_shell
GB.x_ins = scenario.gb_config.tank.x_ins
GB.k_shell = scenario.gb_config.tank.k_shell
GB.k_ins = scenario.gb_config.tank.k_ins
GB.h_o = scenario.gb_config.tank.h_o
GB.system_update()

# Heat Pump Boiler
HPB = HeatPumpBoiler()
HPB.T0 = scenario.hpb_config.temperature.T0
HPB.T_w_tank = scenario.hpb_config.temperature.T_w_tank
HPB.T_w_serv = scenario.hpb_config.temperature.T_w_serv
HPB.T_w_sup = scenario.hpb_config.temperature.T_w_sup
HPB.dV_w_serv = scenario.hpb_config.load.dV_w_serv
HPB.r0 = scenario.hpb_config.tank.r0
HPB.H = scenario.hpb_config.tank.H
HPB.x_shell = scenario.hpb_config.tank.x_shell
HPB.x_ins = scenario.hpb_config.tank.x_ins
HPB.k_shell = scenario.hpb_config.tank.k_shell
HPB.k_ins = scenario.hpb_config.tank.k_ins
HPB.h_o = scenario.hpb_config.tank.h_o
HPB.COP = scenario.hpb_config.COP
HPB.eta_fan = scenario.hpb_config.eta_fan
HPB.dP = scenario.hpb_config.dP
HPB.system_update()

# Extract results as dictionary
eb_results = {
    'E_heater': EB.E_heater,
    'Q_l_tank': EB.Q_l_tank,
    'X_eff': EB.X_eff
}

gb_results = {
    'E_NG': GB.E_NG,
    'Q_exh': GB.Q_exh,
    'X_eff': GB.X_eff
}

hpb_results = {
    'E_cmp': HPB.E_cmp,
    'E_fan_ou': HPB.E_fan,  # HeatPumpBoiler uses E_fan instead of E_fan_ou
    'Q_r_tank': HPB.Q_r_tank,
    'Q_r_ext': HPB.Q_r_ext,
    'X_eff': HPB.X_eff
}

print("Boiler systems initialized and calculated:")
print(f"  EB - E_heater: {eb_results['E_heater']:.2f} W")
print(f"  GB - E_NG: {gb_results['E_NG']:.2f} W")
print(f"  HPB - E_cmp: {hpb_results['E_cmp']:.2f} W, E_fan: {hpb_results['E_fan_ou']:.2f} W")
print(f"  HPB - Total: {hpb_results['E_cmp'] + hpb_results['E_fan_ou']:.2f} W")


Boiler systems initialized and calculated:
  EB - E_heater: 2957.48 W
  GB - E_NG: 3286.09 W
  HPB - E_cmp: 1182.99 W, E_fan: 91.15 W
  HPB - Total: 1274.15 W


### Well-to-Tap 분석 수행

각 국가별로 재생에너지 비율을 스윕하며 well-to-tap 분석을 수행합니다.


In [19]:
# Initialize WellToTapAnalyzer
wtt_analyzer = WellToTapAnalyzer()

# Prepare well-to-tap analysis results storage
wtt_results = {}

# Prepare CO2 factors dictionary for each technology
tech_co2_factors = {
    'coal': DEFAULT_TECH_PARAMS['coal'].co2_emission_factor,
    'gas': DEFAULT_TECH_PARAMS['gas'].co2_emission_factor,
    'nuclear': DEFAULT_TECH_PARAMS['nuclear'].co2_emission_factor,
    'oil': DEFAULT_TECH_PARAMS['oil'].co2_emission_factor,
    'other': DEFAULT_TECH_PARAMS['other'].co2_emission_factor
}

# Handle renewable sub-technologies
if isinstance(DEFAULT_TECH_PARAMS.get('renewable'), dict):
    from grid_mix_analysis import calculate_weighted_renewable_params
    renewable_params = calculate_weighted_renewable_params()
    tech_co2_factors['renewable'] = renewable_params.co2_emission_factor
else:
    tech_co2_factors['renewable'] = DEFAULT_TECH_PARAMS['renewable'].co2_emission_factor

print("Well-to-tap analysis starting...")
print(f"CO2 factors: {tech_co2_factors}")

# Perform well-to-tap analysis for each country and renewable fraction
for country in selected_countries:
    print(f"\nAnalyzing {country}...")
    country_data = COUNTRY_GRID_MIX[country]
    
    wtt_results[country] = []
    
    for ren_frac in renewable_fractions:
        # Calculate generation mix for this renewable fraction
        non_renewable_total = 1.0 - ren_frac
        
        if non_renewable_total > 0:
            base_non_renewable = (
                country_data['coal'] + country_data['gas'] + 
                country_data['nuclear'] + country_data['oil'] + country_data['other']
            )
            scale_factor = non_renewable_total / base_non_renewable
            
            generation_mix = {
                'coal': country_data['coal'] * scale_factor,
                'gas': country_data['gas'] * scale_factor,
                'nuclear': country_data['nuclear'] * scale_factor,
                'renewable': ren_frac,
                'oil': country_data['oil'] * scale_factor,
                'other': country_data['other'] * scale_factor
            }
        else:
            generation_mix = {
                'coal': 0.0, 'gas': 0.0, 'nuclear': 0.0,
                'renewable': 1.0, 'oil': 0.0, 'other': 0.0
            }
        
        # Perform well-to-tap analysis
        wtt_comparison = wtt_analyzer.compare_systems(
            eb_results=eb_results,
            gb_results=gb_results,
            hpb_results=hpb_results,
            generation_mix=generation_mix,
            tech_co2_factors=tech_co2_factors
        )
        
        # Store results
        wtt_results[country].append({
            'renewable_fraction': ren_frac,
            'EB': wtt_comparison['EB'],
            'GB': wtt_comparison['GB'],
            'HPB': wtt_comparison['HPB']
        })
    
    print(f"  Completed {len(wtt_results[country])} data points")

print("\nWell-to-tap analysis completed!")


Well-to-tap analysis starting...
CO2 factors: {'coal': 0.9, 'gas': 0.4, 'nuclear': 0.005, 'oil': 0.7, 'other': 0.5, 'renewable': 0.022000000000000002}

Analyzing South Korea...


KeyError: 'oil'

In [ ]:
selected_countries

['South Korea', 'China', 'Norway']

In [ ]:
# Convert well-to-tap results to DataFrames and save
wtt_dataframes = {}

for country in selected_countries:
    results_list = wtt_results[country]
    
    # Extract data for DataFrame
    data_dict = {
        'renewable_fraction': [r['renewable_fraction'] for r in results_list],
        'EB_primary_energy': [r['EB']['primary_energy_input'] for r in results_list],
        'EB_exergy_gen': [r['EB']['exergy_consumption_generation'] for r in results_list],
        'EB_exergy_grid': [r['EB']['exergy_consumption_grid'] for r in results_list],
        'EB_exergy_total': [r['EB']['exergy_consumption_total'] for r in results_list],
        'EB_co2': [r['EB']['co2_emissions'] for r in results_list],
        'GB_primary_energy': [r['GB']['primary_energy_input'] for r in results_list],
        'GB_exergy_supply': [r['GB']['exergy_consumption_supply'] for r in results_list],
        'GB_exergy_total': [r['GB']['exergy_consumption_total'] for r in results_list],
        'GB_co2': [r['GB']['co2_emissions'] for r in results_list],
        'HPB_primary_energy': [r['HPB']['primary_energy_input'] for r in results_list],
        'HPB_exergy_gen': [r['HPB']['exergy_consumption_generation'] for r in results_list],
        'HPB_exergy_grid': [r['HPB']['exergy_consumption_grid'] for r in results_list],
        'HPB_exergy_total': [r['HPB']['exergy_consumption_total'] for r in results_list],
        'HPB_co2': [r['HPB']['co2_emissions'] for r in results_list]
    }
    
    df_wtt = pd.DataFrame(data_dict)
    wtt_dataframes[country] = df_wtt
    
    # Save to CSV
    csv_path = os.path.join(output_dir, f'{country}_well_to_tap.csv')
    df_wtt.to_csv(csv_path, index=False)
    print(f"Saved well-to-tap results for {country} to {csv_path}")

print("\nAll well-to-tap results saved!")


### Well-to-Tap 결과 시각화

Primary energy input, exergy consumption, CO2 emissions를 비교하는 플롯을 생성합니다.


In [ ]:
# Create well-to-tap comparison plots
fig = plt.figure(
    figsize=(dm.cm2in(17), dm.cm2in(12)),
    dpi=200
)

# GridSpec 레이아웃 설정
gs = fig.add_gridspec(
    nrows=2, ncols=2,
    left=0.12, right=0.95,
    top=0.95, bottom=0.12,
    hspace=0.35, wspace=0.3
)

# 색상 설정
colors = {
    'South Korea': 'oc.blue5',
    'China': 'oc.orange5', 
    'Norway': 'oc.green5'
}
eb_color = 'oc.purple5'
gb_color = 'oc.red5'
hpb_color = 'oc.blue5'
linewidth = 0.7

# 1. Primary Energy Input 비교
ax1 = fig.add_subplot(gs[0, 0])
for country in selected_countries:
    df_wtt = wtt_dataframes[country]
    ax1.plot(df_wtt['renewable_fraction'] * 100, df_wtt['EB_primary_energy'],
            linestyle=':', linewidth=linewidth, color=eb_color, label='EB' if country == selected_countries[0] else '')
    ax1.plot(df_wtt['renewable_fraction'] * 100, df_wtt['GB_primary_energy'],
            linestyle='--', linewidth=linewidth, color=gb_color, label='GB' if country == selected_countries[0] else '')
    ax1.plot(df_wtt['renewable_fraction'] * 100, df_wtt['HPB_primary_energy'],
            linestyle='-', linewidth=linewidth, color=colors[country], label=f'HPB ({country})')

ax1.set_xlabel('Renewable Energy Fraction [%]', fontsize=dm.fs(0))
ax1.set_ylabel('Primary Energy Input [kWh]', fontsize=dm.fs(0))
ax1.set_xlim(0, 100)
ax1.set_xticks([0, 25, 50, 75, 100])
ax1.legend(fontsize=dm.fs(-1), loc='upper right', frameon=True)

# 2. Total Exergy Consumption 비교
ax2 = fig.add_subplot(gs[0, 1])
for country in selected_countries:
    df_wtt = wtt_dataframes[country]
    ax2.plot(df_wtt['renewable_fraction'] * 100, df_wtt['EB_exergy_total'],
            linestyle=':', linewidth=linewidth, color=eb_color, label='EB' if country == selected_countries[0] else '')
    ax2.plot(df_wtt['renewable_fraction'] * 100, df_wtt['GB_exergy_total'],
            linestyle='--', linewidth=linewidth, color=gb_color, label='GB' if country == selected_countries[0] else '')
    ax2.plot(df_wtt['renewable_fraction'] * 100, df_wtt['HPB_exergy_total'],
            linestyle='-', linewidth=linewidth, color=colors[country], label=f'HPB ({country})')

ax2.set_xlabel('Renewable Energy Fraction [%]', fontsize=dm.fs(0))
ax2.set_ylabel('Total Exergy Consumption [kWh]', fontsize=dm.fs(0))
ax2.set_xlim(0, 100)
ax2.set_xticks([0, 25, 50, 75, 100])
ax2.legend(fontsize=dm.fs(-1), loc='upper right', frameon=True)

# 3. CO2 Emissions 비교
ax3 = fig.add_subplot(gs[1, 0])
for country in selected_countries:
    df_wtt = wtt_dataframes[country]
    ax3.plot(df_wtt['renewable_fraction'] * 100, df_wtt['EB_co2'],
            linestyle=':', linewidth=linewidth, color=eb_color, label='EB' if country == selected_countries[0] else '')
    ax3.plot(df_wtt['renewable_fraction'] * 100, df_wtt['GB_co2'],
            linestyle='--', linewidth=linewidth, color=gb_color, label='GB' if country == selected_countries[0] else '')
    ax3.plot(df_wtt['renewable_fraction'] * 100, df_wtt['HPB_co2'],
            linestyle='-', linewidth=linewidth, color=colors[country], label=f'HPB ({country})')

ax3.set_xlabel('Renewable Energy Fraction [%]', fontsize=dm.fs(0))
ax3.set_ylabel('CO$_2$ Emissions [kg CO$_2$]', fontsize=dm.fs(0))
ax3.set_xlim(0, 100)
ax3.set_xticks([0, 25, 50, 75, 100])
ax3.legend(fontsize=dm.fs(-1), loc='upper right', frameon=True)

# 4. Exergy Consumption Breakdown (HPB only, for one country as example)
ax4 = fig.add_subplot(gs[1, 1])
example_country = selected_countries[0]
df_wtt = wtt_dataframes[example_country]
ax4.plot(df_wtt['renewable_fraction'] * 100, df_wtt['HPB_exergy_gen'],
        linestyle='-', linewidth=linewidth, color='oc.blue3', label='Generation')
ax4.plot(df_wtt['renewable_fraction'] * 100, df_wtt['HPB_exergy_grid'],
        linestyle='--', linewidth=linewidth, color='oc.blue4', label='Grid')
ax4.plot(df_wtt['renewable_fraction'] * 100, df_wtt['HPB_exergy_total'],
        linestyle='-', linewidth=linewidth*1.2, color='oc.blue5', label='Total')

ax4.set_xlabel('Renewable Energy Fraction [%]', fontsize=dm.fs(0))
ax4.set_ylabel('Exergy Consumption [kWh]', fontsize=dm.fs(0))
ax4.set_xlim(0, 100)
ax4.set_xticks([0, 25, 50, 75, 100])
ax4.legend(fontsize=dm.fs(-1), loc='upper right', frameon=True)
ax4.set_title(f'HPB Exergy Breakdown ({example_country})', fontsize=dm.fs(0))

# 서브플롯 라벨 추가 (a, b, c, d)
for idx, (ax, label) in enumerate(zip([ax1, ax2, ax3, ax4], 'abcd')):
    offset = dm.make_offset(4, -4, fig)
    ax.text(0, 1, f'({label})', transform=ax.transAxes + offset,
           weight='bold', verticalalignment='top', fontsize=dm.fs(1))

# 레이아웃 최적화
dm.simple_layout(fig)

# 저장 및 표시
os.makedirs('figure', exist_ok=True)
dm.save_formats(
    fig,
    'figure/well_to_tap_comparison',
    formats=('svg', 'png', 'pdf'),
    bbox_inches='tight',
    dpi=300
)
dm.save_and_show(fig, size=600)
